In [64]:
import os
import pickle
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
import cv2  # used for resizing

class MRISliceDataset(Dataset):
    def __init__(self, root_dir, metadata_csv, transform=None, max_files=50, target_size=(320, 320)):
        self.root_dir = root_dir
        self.metadata = pd.read_csv(metadata_csv)
        self.transform = transform
        self.target_size = target_size

        # Keep only first N volumes
        unique_volumes = self.metadata["volumeFilename"].unique()[:max_files]
        self.metadata = self.metadata[self.metadata["volumeFilename"].isin(unique_volumes)]

        self.samples = []
        for _, row in self.metadata.iterrows():
            vol_path = os.path.join(root_dir, row["volumeFilename"])
            if not os.path.exists(vol_path):
                continue
            try:
                with open(vol_path, "rb") as f:
                    volume = pickle.load(f)
                num_slices = volume.shape[0]
            except Exception as e:
                print(f"⚠️ Skipping {vol_path}: {e}")
                continue

            roi_start = int(row["roiZ"])
            roi_end = roi_start + int(row["roiDepth"]) - 1

            for i in range(num_slices):
                label = 1 if roi_start <= i <= roi_end else 0
                self.samples.append((vol_path, i, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        vol_path, slice_idx, label = self.samples[idx]

        with open(vol_path, "rb") as f:
            volume = pickle.load(f)

        image = volume[slice_idx].astype(np.float32)
        image = image / np.max(image) if np.max(image) > 0 else image

        # ✅ Resize to target_size using OpenCV (H, W)
        image = cv2.resize(image, self.target_size, interpolation=cv2.INTER_LINEAR)

        # Add channel dimension
        image = np.expand_dims(image, axis=0)
        image = torch.tensor(image, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label


In [65]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LeNet(nn.Module):
    def __init__(self, para):
        super(LeNet, self).__init__()
        self.para = para
        self.Conv_layer = nn.ModuleList()
        self.BatchNorm_layer = nn.ModuleList()
        self.FC_layer = nn.ModuleList()

        # Build convolution layers immediately
        self.conv_layer_maker()

        # Determine flattened size dynamically
        with torch.no_grad():
            x = torch.randn(1, para['input_channel'], 320, 320)
            x = self.forward_convs(x)
            in_features = x.flatten(1).shape[1]

        # Build FC layers immediately
        self.FC_maker(in_features)

    def conv_layer_maker(self):
        for i in range(self.para['number_of_conv_layer']):
            in_ch = self.para['input_channel'] if i == 0 else self.para['output_channel'][i - 1]
            out_ch = self.para['output_channel'][i]
            conv = nn.Conv2d(
                in_channels=in_ch,
                out_channels=out_ch,
                kernel_size=self.para['Kernel_size'][i],
                stride=self.para['Stride'][i],
                padding=self.para['Padding'][i]
            )
            self.Conv_layer.append(conv)

            if self.para.get('use_batchnorm', False):
                self.BatchNorm_layer.append(nn.BatchNorm2d(out_ch))
            else:
                self.BatchNorm_layer.append(None)

    def FC_maker(self, in_features):
        for i in range(self.para['number_of_FC_layer']):
            in_f = in_features if i == 0 else self.para['FC_features'][i - 1]
            fc = nn.Linear(in_f, self.para['FC_features'][i])
            self.FC_layer.append(fc)

    def choose_activation(self, name):
        if name == 'ReLU':
            return F.relu
        elif name == 'Sigmoid':
            return torch.sigmoid
        elif name == 'Tanh':
            return torch.tanh
        elif name == 'Softmax':
            return lambda x: F.softmax(x, dim=1)
        elif name == 'None' or name is None:
            return lambda x: x
        else:
            raise ValueError(f"Unsupported activation: {name}")

    def choose_pooling(self, name):
        if name == 'MaxPool':
            return F.max_pool2d
        elif name == 'AvgPool':
            return F.avg_pool2d
        else:
            raise ValueError(f"Unsupported pooling: {name}")

    def forward_convs(self, x):
        for i in range(self.para['number_of_conv_layer']):
            act = self.choose_activation(self.para['Activation_Func'][i])
            pool = self.choose_pooling(self.para['Pooling_type'][i])
            x = self.Conv_layer[i](x)
            if self.para.get('use_batchnorm', False) and self.BatchNorm_layer[i] is not None:
                x = self.BatchNorm_layer[i](x)
            x = act(x)
            x = pool(x, 2, 2)
        return x

    def forward(self, x):
        x = self.forward_convs(x)
        x = torch.flatten(x, 1)
        for i in range(self.para['number_of_FC_layer']):
            x = self.FC_layer[i](x)
            act = self.choose_activation(self.para['FC_Activation_Func'][i])
            x = act(x)
        return x

    def summary(self):
        print("LeNet Model")
        for i in range(self.para['number_of_conv_layer']):
            print(f"Conv Layer {i+1}: {self.Conv_layer[i]}")
            if self.para.get('use_batchnorm', False):
                print(f"    BatchNorm: {self.BatchNorm_layer[i]}")
            print(f"    Activation: {self.para['Activation_Func'][i]}")
            print(f"    Pooling: {self.para['Pooling_type'][i]}(kernel_size=2, stride=2)")
        for i in range(self.para['number_of_FC_layer']):
            print(f"FC Layer {i+1}: {self.FC_layer[i]}")
            print(f"    Activation: {self.para['FC_Activation_Func'][i]}")

if __name__ == "__main__":
    para = {
        'input_channel': 1,
        'number_of_conv_layer': 2,
        'number_of_FC_layer': 2,
        'FC_features': [84, 2],
        'output_channel': [6, 16],
        'Kernel_size': [7, 5],
        'Stride': [1, 1],
        'Padding': [0, 0],
        'Activation_Func': ['ReLU', 'ReLU'],
        'Pooling_type': ['MaxPool', 'MaxPool'],
        'FC_Activation_Func': ['ReLU', 'Softmax'], 
        'use_batchnorm': True,
    }
    model = LeNet(para)
    x = torch.randn(1, 1, 320, 320)
    y = model(x)
    print("Output shape:", y.shape)
    model.summary()

Output shape: torch.Size([1, 2])
LeNet Model
Conv Layer 1: Conv2d(1, 6, kernel_size=(7, 7), stride=(1, 1))
    BatchNorm: BatchNorm2d(6, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    Activation: ReLU
    Pooling: MaxPool(kernel_size=2, stride=2)
Conv Layer 2: Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    BatchNorm: BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    Activation: ReLU
    Pooling: MaxPool(kernel_size=2, stride=2)
FC Layer 1: Linear(in_features=92416, out_features=84, bias=True)
    Activation: ReLU
FC Layer 2: Linear(in_features=84, out_features=2, bias=True)
    Activation: Softmax


In [53]:
# class LeNet_Trainer():
#     def __init__(self,model,train_dataloader,test_dataloader,optimizer,epochs,loss_fn,lr,device):
#         self.model=model
#         self.train_dataloader = train_dataloader
#         self.test_dataloader = test_dataloader
#         self.optimizer = optimizer
#         self.epochs= epochs
#         self.loss_fn=loss_fn
#         self.lr=lr
#         self.device=device
#     def train(self):
#         size=len(self.train_dataloader.dataset)
#         self.model.train()
#         for batch,(x,y) in enumerate(self.train_dataloader()):
#             x,y=x.to(self.device),y.to(self.device)
#             pred=self.model(x)
#             loss=self.loss_fn(pred,y)
#             loss.backward()
#             self.optimizer.step()
#             self.optimizer.zero_grad()
#             if batch%10==0:
#                 loss,current=loss.item(),(batch+1)*len(x)
#                 print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

#     def test(self):
#         size=len(self.test_dataloader.dataset)
#         num_batches=len(self.test_dataloader)
#         self.model.eval()
#         test_loss,correct=0,0
#         with torch.no_grad():
#             for x,y in self.test_dataloader:
#                 x,y=x.to(self.device),y.to(self.device)
#                 pred=self.model(x)
#                 test_loss+=self.loss_fn(pred,y).item()
#                 correct += (pred.argmax(1) == y).type(torch.float).sum().item()
#         test_loss /= num_batches
#         correct /= size
#         print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


In [66]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

class Trainer:
    def __init__(self, model, train_dataset, val_dataset=None,
                 batch_size=512, lr=1e-4, num_workers=0, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)

        # Dataloaders
        self.train_loader = DataLoader(train_dataset, batch_size=batch_size,
                                       shuffle=True, num_workers=num_workers)
        self.val_loader = None
        if val_dataset is not None:
            self.val_loader = DataLoader(val_dataset, batch_size=batch_size,
                                         shuffle=True, num_workers=num_workers)

        # Loss and optimizer
        self.criterion = nn.CrossEntropyLoss()  # binary classification
        self.optimizer = torch.optim.SGD(self.model.parameters(),lr=lr)

    def train_epoch(self):
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        loop = tqdm(self.train_loader, desc="Training", leave=False)
        for images, labels in loop:
            images, labels = images.to(self.device), labels.to(self.device)
            
            # Forward
            outputs = self.model(images).squeeze(1)  # [B] or [B,1]
            loss = self.criterion(outputs, labels)

            # Backward
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            # Metrics
            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels.long()).sum().item()
            total += labels.size(0)
            loop.set_postfix(loss=loss.item())

        avg_loss = running_loss / total
        acc = correct / total
        return avg_loss, acc

    def validate_epoch(self):
        if self.val_loader is None:
            return None, None

        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            loop = tqdm(self.val_loader, desc="Validating", leave=False)
            for images, labels in loop:
                images, labels = images.to(self.device), labels.to(self.device)
                outputs = self.model(images).squeeze(1)
                loss = self.criterion(outputs, labels)
                print(outputs.shape,outputs.dtype)
                print(labels.shape,labels.dtype)
                running_loss += loss.item() * images.size(0)
                preds = torch.argmax(outputs, dim=1)
                correct += (preds == labels.long()).sum().item()
                total += labels.size(0)

        avg_loss = running_loss / total
        acc = correct / total
        return avg_loss, acc

    def fit(self, epochs=2):
        print(f"Training on device: {self.device}")
        for epoch in range(1, epochs + 1):
            train_loss, train_acc = self.train_epoch()
            val_loss, val_acc = self.validate_epoch() if self.val_loader else (None, None)

            msg = f"Epoch [{epoch}/{epochs}] | Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}"
            if val_loss is not None:
                msg += f" | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
            print(msg)


In [55]:
print(torch.cuda.is_available())

True


In [68]:
from torch.utils.data import random_split

full_dataset = MRISliceDataset("volumetric_data/", "metadata.csv")
train_size = int(0.7 * len(full_dataset))
print(train_size)
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])
para = {
        'input_channel': 1,
        'number_of_conv_layer': 2,
        'number_of_FC_layer': 2,
        'FC_features': [84, 2],
        'output_channel': [6, 16],
        'Kernel_size': [7, 5],
        'Stride': [1, 1],
        'Padding': [0, 0],
        'Activation_Func': ['ReLU', 'ReLU'],
        'Pooling_type': ['MaxPool', 'MaxPool'],
        'FC_Activation_Func': ['ReLU', 'None'], 
        'use_batchnorm': True,
    }
model = LeNet(para)
print(sum(p.numel() for p in model.parameters()))  # should be > 0
trainer = Trainer(model, train_ds, val_ds, batch_size=8, lr=1e-4)
trainer.fit(epochs=10)


1019
7765958
Training on device: cuda


Validating:   4%|▎         | 2/55 [00:00<00:05,  9.87it/s]              

torch.Size([8, 2]) torch.float32
torch.Size([8]) torch.int64
torch.Size([8, 2]) torch.float32
torch.Size([8]) torch.int64


Validating:   7%|▋         | 4/55 [00:00<00:05,  9.33it/s]

torch.Size([8, 2]) torch.float32
torch.Size([8]) torch.int64
torch.Size([8, 2]) torch.float32
torch.Size([8]) torch.int64


...

Validating:  91%|█████████ | 50/55 [00:05<00:00,  9.72it/s]

torch.Size([8, 2]) torch.float32
torch.Size([8]) torch.int64
torch.Size([8, 2]) torch.float32
torch.Size([8]) torch.int64


Validating:  93%|█████████▎| 51/55 [00:05<00:00,  9.60it/s]

torch.Size([8, 2]) torch.float32
torch.Size([8]) torch.int64
torch.Size([8, 2]) torch.float32
torch.Size([8]) torch.int64


torch.Size([8, 2]) torch.float32
torch.Size([8]) torch.int64
torch.Size([8, 2]) torch.float32
torch.Size([8]) torch.int64
torch.Size([6, 2]) torch.float32
torch.Size([6]) torch.int64
Epoch [10/10] | Train Loss: 0.0966, Acc: 0.9715 | Val Loss: 0.1544, Val Acc: 0.9315
